<a href="https://colab.research.google.com/github/marinhomar/blank-app/blob/main/parte1_extracao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# pandas: para trabalhar com tabelas (como uma planilha dentro do Python)
import pandas as pd

# unicodedata: para tirar os acentos das palavras
import unicodedata

In [ ]:
# endereço da pasta onde estão os arquivos no GitHub
BASE = "https://raw.githubusercontent.com/marinhomar/cardioia-fase2/main/src/parte1/"

# frases dos pacientes (uma por linha)
# sep="\t" porque as frases têm vírgula e não podem ser cortadas
frases = pd.read_csv(BASE + "frases.txt", header=None, names=["frase"], sep="\t")

# mapa: qual sintoma aponta para qual doença
mapa = pd.read_csv(BASE + "mapa_conhecimento.csv")

# sinônimos: jeitos populares de falar o mesmo sintoma
sinonimos = pd.read_csv(BASE + "sinonimos.csv")

# mostra as frases para conferir se carregou certo
frases

In [4]:
# deixa o texto em minúsculo e sem acento
# assim "Tórax", "torax" e "TÓRAX" viram a mesma coisa
def normalizar(texto):
    texto = texto.lower()                          # tudo minúsculo
    texto = unicodedata.normalize("NFD", texto)    # separa a letra do acento
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")  # joga o acento fora

In [5]:
# coloca os sinônimos no mesmo padrão (minúsculo e sem acento)
sinonimos["variante"] = sinonimos["variante"].apply(normalizar)
sinonimos["termo_padrao"] = sinonimos["termo_padrao"].apply(normalizar)

# ordena do maior para o menor
# assim "pernas inchada" é trocada antes de "perna inchada"
sinonimos = sinonimos.sort_values("variante", key=lambda s: s.str.len(), ascending=False)

# troca o jeito popular pelo termo que está no mapa
# exemplo: "agonia no peito" vira "dor no peito"
def traduzir(texto):
    for variante, padrao in zip(sinonimos["variante"], sinonimos["termo_padrao"]):
        texto = texto.replace(variante, padrao)
    return texto

In [6]:
# ordem de gravidade: em caso de empate, vale a mais grave (como na triagem do pronto-socorro)
GRAVIDADE = ["Infarto", "Angina", "Insuficiência Cardíaca"]

def diagnosticar(frase):
    # prepara a frase: padroniza e traduz os sinônimos
    texto = traduzir(normalizar(frase))

    pontos = {}        # quantos sintomas cada doença recebeu
    encontrados = []   # quais sintomas apareceram na frase

    # passa por cada linha do mapa procurando os sintomas na frase
    for _, linha in mapa.iterrows():
        for sintoma in (linha["Sintoma 1"], linha["Sintoma 2"]):
            if normalizar(sintoma) in texto:
                encontrados.append(sintoma)
                doenca = linha["Doença Associada"]
                pontos[doenca] = pontos.get(doenca, 0) + 1   # +1 ponto para essa doença

    # nenhum sintoma encontrado
    if not pontos:
        return encontrados, "Sem diagnóstico"

    # pega a doença (ou doenças) com mais pontos
    maior = max(pontos.values())
    empatadas = [d for d in pontos if pontos[d] == maior]

    # só uma ganhou: esse é o diagnóstico
    if len(empatadas) == 1:
        return encontrados, empatadas[0]

    # empate: avisa e sugere a mais grave
    prioridade = min(empatadas, key=GRAVIDADE.index)
    return encontrados, f"Inconclusivo ({' / '.join(empatadas)}) → priorizar: {prioridade}"

In [ ]:
# copia a tabela de frases para não mexer na original
resultado = frases.copy()

# aplica o diagnóstico em cada frase e cria duas colunas novas
resultado[["sintomas_encontrados", "diagnostico"]] = resultado["frase"].apply(
    lambda f: pd.Series(diagnosticar(f))
)

# deixa mostrar a frase inteira na tabela
pd.set_option("display.max_colwidth", 120)

# mostra o resultado final
resultado

In [ ]:
# frase nova, com expressões do dia a dia que não estão no mapa
# os sinônimos fazem a ponte e o sistema entende mesmo assim
diagnosticar("Desde ontem tô com uma agonia no peito e suando frio, não fui trabalhar.")